In [ ]:
import time
import pandas as pd
import numpy as np
import umap
import plotly.graph_objects as go
from pathlib import Path
from sklearn.manifold import MDS

# -------------------------
# 1. Début du script - Mesure de temps
# -------------------------
start_time = time.time()

# -------------------------
# 2. Paramètres et chemins
# -------------------------
MDS_DIM = 60  # Nombre de dimensions pour MDS
data_dir = Path("../data")
jaccard_file = Path("../data/jaccard_overall.parquet")  # Fichier des distances de Jaccard
journals_file = Path("journals_info.csv")              # Informations complémentaires sur les journaux

# Chemins pour la sauvegarde des résultats intermédiaires et finaux
processed_df_path = Path("./data/processed_embedding_df.parquet")
mds_save_path = Path("./data/mds_embeddings.npy")
umap_save_path = Path("./data/umap_embeddings.npy")

# -------------------------
# 3. Chargement des données ou calcul si sauvegarde inexistante
# -------------------------
if processed_df_path.exists() and mds_save_path.exists() and umap_save_path.exists():
    print("Chargement des résultats sauvegardés...")
    embedding_df = pd.read_parquet(processed_df_path)
    mds_embeddings = np.load(mds_save_path)
    embedded_journals = np.load(umap_save_path)
    # Vous pouvez recharger aussi d'éventuelles autres données (ex. couleur, discipline) si nécessaire.
else:
    print("Aucun fichier sauvegardé trouvé, début du calcul complet...")
    
    # Lecture du fichier Parquet contenant la matrice Jaccard
    print("Chargement de la matrice Jaccard...")
    df = pd.read_parquet(jaccard_file)
    
    # Conversion en numérique (attention : évitez les copies inutiles)
    df = df.apply(pd.to_numeric, errors='coerce').astype(np.float32)
    df.fillna(1, inplace=True)
    # Une seconde conversion si nécessaire (mais à éviter si redondante)
    df = df.apply(pd.to_numeric, errors='coerce')
    df.fillna(1, inplace=True)
    
    # Forcer la symétrie de la matrice
    if not np.allclose(df.values, df.values.T, atol=1e-5):
        print("La matrice n'est pas symétrique, on la symétrise...")
        df = (df + df.T) / 2
    else:
        print("La matrice est déjà symétrique.")
    
    # -------------------------
    # 4. Application de la normalisation
    #    Formule : f(x) = 1 - (1/(1 - log(1-x)))
    # -------------------------
    print("Application de la normalisation : 1 - (1/(1-log(1-x))) sur la matrice Jaccard...")
    mask = df < 1
    df_norm = df.copy()
    df_norm[mask] = 1 - (1 / (1 - np.log(1 - df[mask])))
    # Pour x = 1 (log(0) non défini), on remplace par une constante (double de la valeur max calculée)
    max_non1 = df_norm[mask].max().max()
    df_norm[~mask] = max_non1 * 2
    print("Transformation normalisée appliquée.")
    
    # -------------------------
    # 5. Réduction de dimension avec MDS
    # -------------------------
    print(f"Réduction de dimension avec MDS ({MDS_DIM} dimensions) en cours...")
    mds = MDS(n_components=MDS_DIM, dissimilarity="precomputed",
              random_state=42, normalized_stress="auto", n_jobs=-1)
    mds_embeddings = mds.fit_transform(df_norm.values)
    print("Réduction MDS terminée.")
    # Sauvegarde des embeddings MDS
    np.save(mds_save_path, mds_embeddings)
    
    # -------------------------
    # 6. Projection UMAP sur les embeddings MDS
    # -------------------------
    print("Projection UMAP en cours...")
    reducer = umap.UMAP(metric='euclidean', random_state=42, n_components=2)
    embedded_journals = reducer.fit_transform(mds_embeddings)
    print("Projection UMAP terminée.")
    # Sauvegarde des embeddings UMAP
    np.save(umap_save_path, embedded_journals)
    
    # -------------------------
    # 7. Chargement des données des journaux depuis le fichier CSV
    # -------------------------
    print("Chargement du fichier CSV avec les informations des journaux...")
    journal_df = pd.read_csv(journals_file)
    # On filtre pour ne conserver que les journaux présents dans la matrice (index de df)
    journal_df = journal_df[journal_df["journal_id"].isin(df.index)]
    # Création des mappings pour le nom et la discipline
    journal_names = dict(zip(journal_df["journal_id"], journal_df["name"]))
    journal_discipline = dict(zip(journal_df["journal_id"], journal_df["discipline"]))
    discipline_colors = {
        "AGRI": "rgb(102,194,165)", "ARTS": "rgb(252,141,98)", "BIOC": "rgb(141,160,203)", 
        "BUSI": "rgb(231,138,195)", "CENG": "rgb(166,216,84)", "CHEM": "rgb(255,217,47)", 
        "COMP": "rgb(229,196,148)", "DECI": "rgb(179,179,179)", "DENT": "rgb(228,26,28)", 
        "EART": "rgb(55,126,184)", "ECON": "rgb(77,175,74)", "ENER": "rgb(152,78,163)", 
        "ENGI": "rgb(255,127,0)", "ENVI": "rgb(255,255,51)", "HEAL": "rgb(166,86,40)", 
        "IMMU": "rgb(247,129,191)", "MATE": "rgb(153,153,153)", "MATH": "rgb(141,211,199)", 
        "MEDI": "rgb(255,255,179)", "MULT": "rgb(190,186,218)", "NEUR": "rgb(251,128,114)", 
        "NURS": "rgb(128,177,211)", "PHAR": "rgb(253,180,98)", "PHYS": "rgb(179,222,105)", 
        "PSYC": "rgb(252,205,229)", "SOCI": "rgb(217,217,217)", "VETE": "rgb(188,128,189)"
    }
    journal_colors = {jid: discipline_colors.get(journal_discipline.get(jid, "SOCI"), "black")
                      for jid in journal_names.keys()}
    
    # -------------------------
    # 8. Création de l'Embedding DataFrame
    # -------------------------
    embedding_df = pd.DataFrame(embedded_journals, columns=["UMAP-1", "UMAP-2"])
    embedding_df["id"] = df.index
    embedding_df["name"] = embedding_df["id"].map(journal_names).fillna(embedding_df["id"])
    embedding_df["color"] = embedding_df["id"].map(journal_colors).fillna("black")
    embedding_df["discipline"] = embedding_df["id"].map(journal_discipline).fillna("SOCI")
    
    # Sauvegarde du DataFrame traité
    embedding_df.to_parquet(processed_df_path)
    print(f"DataFrame traité sauvegardé dans {processed_df_path}")

# -------------------------
# 9. Calcul des médoïdes
# -------------------------
print("📌 Calcul des médoïdes...")
def compute_medoid(points):
    dists = np.linalg.norm(points[:, None, :] - points[None, :, :], axis=2)
    return points[np.argmin(dists.sum(axis=1))]

representative_points = {}
# On parcourt chaque discipline présente dans le DataFrame
for disc in embedding_df["discipline"].unique():
    pts = embedding_df[embedding_df["discipline"] == disc][["UMAP-1", "UMAP-2"]].values
    if len(pts) > 0:
        representative_points[disc] = compute_medoid(pts)

# -------------------------
# 10. Visualisation Plotly
# -------------------------
print("🎨 Affichage des points UMAP + médoïdes...")
fig = go.Figure()

# Ajout des points correspondant aux journaux
for discipline, color in discipline_colors.items():
    subset = embedding_df[embedding_df["discipline"] == discipline]
    fig.add_trace(go.Scattergl(
        x=subset["UMAP-1"],
        y=subset["UMAP-2"],
        mode="markers",
        marker=dict(color=color, size=5),
        text=subset["name"],
        hoverinfo="text",
        name=discipline
    ))

# Ajout des médoïdes pour chaque discipline (affichés sous forme de croix)
for disc, coord in representative_points.items():
    color = discipline_colors.get(disc, "black")
    fig.add_trace(go.Scattergl(
        x=[coord[0]],
        y=[coord[1]],
        mode="markers+text",
        marker=dict(color=color, size=14, symbol="x"),
        text=[disc],
        textposition="middle right",
        name="Médoïde: " + disc
    ))

fig.update_layout(
    title="🧬 Projection UMAP avec Médoïdes colorés par discipline",
    showlegend=True
)
fig.write_html("umap_medoid_only.html")
fig.show()

# -------------------------
# Fin du script
# -------------------------
print("✅ Terminé sans alignement !")
print(f"⏱️ Temps d'exécution : {time.time() - start_time:.2f} secondes")

/home/smeziou/Bureau/multdisciplinaryOnlineTool/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Aucun fichier sauvegardé trouvé, début du calcul complet...
Chargement de la matrice Jaccard...
La matrice est déjà symétrique.
Application de la normalisation : 1 - (1/(1-log(1-x))) sur la matrice Jaccard...
Transformation normalisée appliquée.
Réduction de dimension avec MDS (60 dimensions) en cours...


In [ ]:
from pathlib import Path
import json
import pandas as pd
from collections import defaultdict, Counter
import pyarrow as pa
import pyarrow.parquet as pq

print("🚀 Début du traitement incrémental avec fusion...")

# Chemins
base_dir = Path("../data/raw data/raw/scopus")
parquet_output_path = Path("./researchers_publications.parquet")

# Disciplines
disciplines = [
    "AGRI", "ARTS", "BIOC", "BUSI", "CENG", "CHEM", "COMP", "DECI", "DENT",
    "EART", "ECON", "ENER", "ENGI", "ENVI", "HEAL", "IMMU", "MATE", "MATH",
    "MEDI", "MULT", "NEUR", "NURS", "PHAR", "PHYS", "PSYC", "SOCI", "VETE"
]

# Chargement initial si fichier existe
if parquet_output_path.exists():
    print(f"📥 Chargement du fichier existant : {parquet_output_path}")
    df_authors = pd.read_parquet(parquet_output_path)
    author_map = {
        row["author_name"]: {
            "journals": {j["journal_id"]: j["count"] for j in row["journals"]},
            "disciplines": Counter([row["dominant_discipline"]])
        }
        for _, row in df_authors.iterrows()
    }
else:
    author_map = {}

# Parcours discipline par discipline
for discipline in disciplines:
    disc_dir = base_dir / discipline
    if not disc_dir.exists():
        print(f"⚠️ Dossier non trouvé : {disc_dir}")
        continue

    print(f"📂 Discipline : {discipline}")
    files = list(disc_dir.glob("*.json"))
    print(f"   → {len(files)} fichiers")

    for file_path in files:
        try:
            with open(file_path, encoding="utf-8") as f:
                data = json.load(f)
                for entry in data.get("entry", []):
                    journal_id = entry.get("source-id")
                    if not journal_id:
                        continue
                    for author in entry.get("author", []):
                        name = author.get("authname")
                        if not name:
                            continue

                        if name not in author_map:
                            author_map[name] = {"journals": defaultdict(str), "disciplines": Counter()}

                        author_map[name]["journals"][journal_id] += 1
                        author_map[name]["disciplines"][discipline] += 1

        except Exception as e:
            print(f"   ❌ Erreur {file_path.name} : {e}")

    print(f"   ✔️ Données fusionnées pour discipline {discipline}")

# Conversion en DataFrame final
print("📦 Conversion en DataFrame...")
records = []
for name, data in author_map.items():
    journal_list = [{"journal_id": jid, "count": count} for jid, count in data["journals"].items()]
    dominant_disc = data["disciplines"].most_common(1)[0][0]
    records.append({
        "author_name": name,
        "journals": journal_list,
        "dominant_discipline": dominant_disc
    })

df_final = pd.DataFrame(records)

# Écriture Parquet
print(f"💾 Écriture du fichier Parquet ({len(df_final)} auteurs)...")
df_final.to_parquet(parquet_output_path, index=False, engine="pyarrow")
print("✅ Terminé ! Tout est à jour dans :", parquet_output_path)


🚀 Début du traitement incrémental avec fusion...
📥 Chargement du fichier existant : researchers_publications.parquet
📂 Discipline : AGRI
   → 65 fichiers
   ✔️ Données fusionnées pour discipline AGRI
📂 Discipline : ARTS
   → 65 fichiers
   ✔️ Données fusionnées pour discipline ARTS
📂 Discipline : BIOC
   → 65 fichiers
   ✔️ Données fusionnées pour discipline BIOC
📂 Discipline : BUSI
   → 65 fichiers
   ✔️ Données fusionnées pour discipline BUSI
📂 Discipline : CENG
   → 65 fichiers
   ✔️ Données fusionnées pour discipline CENG
📂 Discipline : CHEM
   → 65 fichiers
   ✔️ Données fusionnées pour discipline CHEM
📂 Discipline : COMP
   → 72 fichiers
   ✔️ Données fusionnées pour discipline COMP
📂 Discipline : DECI
   → 65 fichiers
   ✔️ Données fusionnées pour discipline DECI
📂 Discipline : DENT
   → 65 fichiers
   ✔️ Données fusionnées pour discipline DENT
📂 Discipline : EART
   → 65 fichiers
   ✔️ Données fusionnées pour discipline EART
📂 Discipline : ECON
   → 65 fichiers
   ✔️ Données fu

In [1]:
from pathlib import Path
import json
import pandas as pd
from collections import defaultdict
import pyarrow.parquet as pq

# 🔧 Fichiers
umap_path = Path("../data/umap_positions.parquet")
output_path = Path("../data/umap_positions_with_source_id.parquet")
raw_data_dir = Path("../data/raw data/raw/scopus")

# Chargement du fichier UMAP
df_umap = pd.read_parquet(umap_path)

# Normalisation des noms de journaux (pour le matching)
def normalize_name(name: str) -> str:
    return name.lower().replace("’", "'").strip()

df_umap["normalized_name"] = df_umap["name"].apply(normalize_name)

# 📦 Dictionnaire nom_journal → source_id
journal_to_sourceid = {}

# Parcours des fichiers JSON
for json_file in raw_data_dir.rglob("*.json"):
    try:
        with open(json_file, encoding="utf-8") as f:
            data = json.load(f)
            for entry in data.get("entry", []):
                name = entry.get("prism:publicationName")
                source_id = entry.get("source-id")
                if name and source_id:
                    norm_name = normalize_name(name)
                    if norm_name not in journal_to_sourceid:
                        journal_to_sourceid[norm_name] = source_id
    except Exception as e:
        print(f"⚠️ Erreur avec {json_file}: {e}")

print(f"🔍 Source-IDs extraits pour {len(journal_to_sourceid)} journaux")

# Ajout de la colonne source-id dans le DataFrame UMAP
df_umap["source-id"] = df_umap["normalized_name"].map(journal_to_sourceid)

# Suppression de la colonne temporaire
df_umap.drop(columns=["normalized_name"], inplace=True)

# 💾 Sauvegarde du nouveau fichier
df_umap.to_parquet(output_path, index=False)
print(f"✅ Fichier enrichi sauvegardé dans : {output_path}")


🔍 Source-IDs extraits pour 94956 journaux
✅ Fichier enrichi sauvegardé dans : ../data/umap_positions_with_source_id.parquet


In [1]:
# scripts/compute_researchers_barycentres.py

import numpy as np
import pandas as pd
from pathlib import Path
from unidecode import unidecode

# 1) Chemins
BASE_DIR = Path("../data")
RESEARCHERS_FILE = BASE_DIR / "researchers_publications.parquet"
COORDS_FILE     = BASE_DIR / "umap_positions_with_source_id.parquet"
OUT_FILE        = BASE_DIR / "researchers_barycentres.parquet"

# 2) Chargement
researchers = pd.read_parquet(RESEARCHERS_FILE)
coords      = pd.read_parquet(COORDS_FILE)

# 3) Prépare les coords pour jointure
if "id" in coords.columns and "source-id" not in coords.columns:
    coords = coords.rename(columns={"id": "source-id"})
coords["source-id"] = coords["source-id"].astype(str)
coords = coords[["source-id", "UMAP-1", "UMAP-2"]]

# 4) Fonction de normalisation
norm = lambda s: unidecode(s).casefold().strip()

# 5) Calcule tous les barycentres
rows = []
for _, row in researchers.iterrows():
    auth = row["author_name"]
    raw  = row["journals"]
    if not isinstance(raw, (list, np.ndarray)):
        continue
    # extrait [(source-id, count), …]
    entries = []
    for it in raw:
        if not isinstance(it, dict) or not it.get("journal_id"):
            continue
        sid = str(it["journal_id"])
        try:
            cnt = max(int(it.get("count", 1)), 1)
        except:
            cnt = 1
        entries.append({"source-id": sid, "count": cnt})
    if not entries:
        continue

    df_tmp = pd.DataFrame(entries)
    merged = df_tmp.merge(coords, on="source-id", how="inner")
    if merged.empty:
        continue

    w = merged["count"].to_numpy()[:, None]
    v = merged[["UMAP-1","UMAP-2"]].to_numpy()
    bary = (w * v).sum(axis=0) / w.sum()

    rows.append({
        "author_name": auth,
        "norm_name":   norm(auth),
        "x":           float(bary[0]),
        "y":           float(bary[1]),
    })

# 6) Sauvegarde
bary_df = pd.DataFrame(rows)
bary_df.to_parquet(OUT_FILE, index=False)
print(f"✅ {len(bary_df)} barycentres calculés et sauvés dans {OUT_FILE}")


✅ 1809211 barycentres calculés et sauvés dans ../data/researchers_barycentres.parquet


In [ ]:
# scripts/compute_researchers_barycentres.py

import numpy as np
import pandas as pd
from pathlib import Path
from unidecode import unidecode

# quand on est en notebook ou REPL
project_root = Path("..")
BASE_DIR         = project_root / "data"
RESEARCHERS_FILE = BASE_DIR / "researchers_publications.parquet"
COORDS_FILE      = BASE_DIR / "umap_positions_with_source_id.parquet"
OUT_FILE         = BASE_DIR / "researchers_barycentres.parquet"
# 2) Chargement
researchers = pd.read_parquet(RESEARCHERS_FILE)
coords      = pd.read_parquet(COORDS_FILE)

# 3) Prépare les coords pour jointure
if "id" in coords.columns and "source-id" not in coords.columns:
    coords = coords.rename(columns={"id": "source-id"})
coords["source-id"] = coords["source-id"].astype(str)
coords = coords[["source-id", "UMAP-1", "UMAP-2"]]

# 4) Fonction de normalisation
norm = lambda s: unidecode(s).casefold().strip()

# 5) Calcule tous les barycentres
rows = []
for _, row in researchers.iterrows():
    auth = row["author_name"]
    raw  = row["journals"]
    if not isinstance(raw, (list, np.ndarray)):
        continue
    # extrait [(source-id, count), …]
    entries = []
    for it in raw:
        if not isinstance(it, dict) or not it.get("journal_id"):
            continue
        sid = str(it["journal_id"])
        try:
            cnt = max(int(it.get("count", 1)), 1)
        except:
            cnt = 1
        entries.append({"source-id": sid, "count": cnt})
    if not entries:
        continue

    df_tmp = pd.DataFrame(entries)
    merged = df_tmp.merge(coords, on="source-id", how="inner")
    if merged.empty:
        continue

    w = merged["count"].to_numpy()[:, None]
    v = merged[["UMAP-1","UMAP-2"]].to_numpy()
    bary = (w * v).sum(axis=0) / w.sum()

    rows.append({
        "author_name": auth,
        "norm_name":   norm(auth),
        "x":           float(bary[0]),
        "y":           float(bary[1]),
    })

# 6) Sauvegarde
bary_df = pd.DataFrame(rows)
bary_df.to_parquet(OUT_FILE, index=False)
print(f"✅ {len(bary_df)} barycentres calculés et sauvés dans {OUT_FILE}")
